# PFE Spark — Fine-tuning v3 avec has_parent

**Nouveautés vs v2 :**
- Feature `has_parent` récupérée via API JIRA → signal Sub-task parfait
- 4 classes : Bug / Improvement / Sub-task / Other
- Texte enrichi avec `[HAS-PARENT]` / `[NO-PARENT]`

**Objectif** : >70% accuracy

In [1]:
# ── Cellule 1 : Install ───────────────────────────────────────────────────
import subprocess
subprocess.run('pip install -q transformers==4.46.3 datasets sentencepiece scikit-learn imbalanced-learn', shell=True)
print('OK')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 91.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 102.8 MB/s eta 0:00:00
OK


In [2]:
# ── Cellule 2 : GPU check ────────────────────────────────────────────────
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('GPU :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
x = torch.ones(2,2).to(DEVICE)
print('CUDA test OK :', (x+1).sum().item())

GPU : Tesla T4
CUDA test OK : 8.0


In [4]:
# ── Cellule 3 : Charger les données ──────────────────────────────────────
import pandas as pd
import numpy as np

# Chemins Kaggle — ajouter les 2 datasets dans Settings > Add Data
CSV_MAIN   = '/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/mart-ml-export-csv/mart_ml_export.csv'
CSV_PARENT = '/kaggle/input/datasets/YOUR_KAGGLE_USERNAME/spark-parent-key/spark_parent_keys.csv'

df      = pd.read_csv(CSV_MAIN)
parents = pd.read_csv(CSV_PARENT)[['key', 'has_parent']]

# Normaliser les noms de colonnes en minuscules
df.columns      = df.columns.str.lower()
parents.columns = parents.columns.str.lower()

print('Colonnes df     :', list(df.columns))
print('Colonnes parents:', list(parents.columns))

# Jointure
df = df.merge(parents, on='key', how='left')
df['has_parent'] = df['has_parent'].fillna(0).clip(0, 1).astype(int)

print(f'Tickets : {len(df):,}')
print(f'has_parent=1 : {df["has_parent"].sum():,} ({df["has_parent"].mean()*100:.1f}%)')

# --- Remapping 4 classes ---
def remap(x):
    if x in ['Task', 'Documentation', 'New Feature']: return 'Other'
    if x in ['Test', 'Question']:                      return 'Other'
    return x  # Bug, Improvement, Sub-task, Other inchangés

df['issuetype'] = df['issuetype'].apply(remap)

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'validation'].reset_index(drop=True)

CLASSES  = ['Bug', 'Improvement', 'Other', 'Sub-task']
label2id = {l: i for i, l in enumerate(CLASSES)}
id2label = {i: l for i, l in enumerate(CLASSES)}

print('\nDistribution train :')
print(train_df['issuetype'].value_counts())
print('\nhas_parent par issuetype (train) :')
print(train_df.groupby('issuetype')['has_parent'].mean().mul(100).round(1))

Colonnes df     : ['key', 'split', 'issuetype', 'summary_clean', 'description_clean', 'text_for_it', 'n_comments', 'n_total_changes', 'description_length', 'summary_length', 'n_links_total', 'was_escalated']
Colonnes parents: ['key', 'has_parent']
Tickets : 42,083
has_parent=1 : 8,841 (21.0%)

Distribution train :
issuetype
Bug            15037
Improvement    11157
Sub-task        7277
Other           4803
Name: count, dtype: int64

has_parent par issuetype (train) :
issuetype
Bug              0.0
Improvement      0.0
Other            0.1
Sub-task       100.0
Name: has_parent, dtype: float64


In [5]:
# ── Cellule 4 : Texte enrichi ─────────────────────────────────────────────
# [HAS-PARENT] est le signal clé pour Sub-task
# [NO-PARENT]  renforce Bug/Improvement/Other

def build_text(row):
    text    = str(row.get('text_for_it') or row.get('summary_clean') or '')
    summary = str(row.get('summary_clean') or '').lower()

    # Signal parent — le plus important
    parent_flag = '[HAS-PARENT] ' if row['has_parent'] == 1 else '[NO-PARENT] '

    # Signal keyword
    kw = ''
    if any(k in summary for k in ['fix','bug','error','fail','crash','exception','npe']):
        kw = '[BUG-SIGNAL] '
    elif any(k in summary for k in ['improve','enhance','optimize','performance','refactor']):
        kw = '[IMPROVEMENT-SIGNAL] '

    return (parent_flag + kw + text)[:512]

train_df['text'] = train_df.apply(build_text, axis=1)
val_df['text']   = val_df.apply(build_text, axis=1)

# Vérification
subtasks = train_df[train_df['issuetype'] == 'Sub-task']
print(f'Sub-tasks avec [HAS-PARENT] : {subtasks["text"].str.startswith("[HAS-PARENT]").mean()*100:.1f}%')
print()
print('Exemples Sub-task :')
for t in subtasks['text'].head(3):
    print(' ', t[:100])

Sub-tasks avec [HAS-PARENT] : 100.0%

Exemples Sub-task :
  [HAS-PARENT] TICKET: Update `NAMESPACE` file in SparkR for simple parameters functions
PRI: Major
DE
  [HAS-PARENT] TICKET: ML model broadcasts should be stored in private vars: spark.ml tree ensembles
P
  [HAS-PARENT] [SHORT-DESC] TICKET: ML model broadcasts should be stored in private vars: spark.ml Wor


In [6]:
# ── Cellule 5 : Tokenisation ─────────────────────────────────────────────
from transformers import AutoTokenizer

MODEL_NAME = 'microsoft/deberta-v3-base'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(texts, padding='max_length', truncation=True, max_length=256)

print('Tokenisation train...')
train_enc = tokenize(train_df['text'].tolist())
print('Tokenisation val...')
val_enc   = tokenize(val_df['text'].tolist())
print('OK')

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenisation train...
Tokenisation val...
OK


In [7]:
# ── Cellule 6 : Dataset PyTorch ───────────────────────────────────────────
import torch
from torch.utils.data import Dataset, DataLoader

class JiraDataset(Dataset):
    def __init__(self, encodings, labels):
        self.input_ids      = torch.tensor(encodings['input_ids'],      dtype=torch.long)
        self.attention_mask = torch.tensor(encodings['attention_mask'], dtype=torch.long)
        self.labels         = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {'input_ids': self.input_ids[i],
                'attention_mask': self.attention_mask[i],
                'labels': self.labels[i]}

train_labels = [label2id[l] for l in train_df['issuetype']]
val_labels   = [label2id[l] for l in val_df['issuetype']]

train_ds     = JiraDataset(train_enc, train_labels)
val_ds       = JiraDataset(val_enc,   val_labels)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds):,}  Val: {len(val_ds):,}')

Train: 38,274  Val: 3,809


In [8]:
# ── Cellule 7 : Modèle + class weights ───────────────────────────────────
import torch.nn as nn
from transformers import AutoModelForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(CLASSES),
    id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float32,
).to(DEVICE)

cw      = compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=train_labels)
cw_t    = torch.tensor(cw, dtype=torch.float).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(weight=cw_t)

print('Class weights:', {CLASSES[i]: round(float(w),2) for i,w in enumerate(cw)})

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
pooler.dense.bias        

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Class weights: {'Bug': 0.64, 'Improvement': 0.86, 'Other': 1.99, 'Sub-task': 1.31}


In [9]:
# ── Cellule 8 : Optimizer ────────────────────────────────────────────────
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

N_EPOCHS    = 5
total_steps = len(train_loader) * N_EPOCHS
optimizer   = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler   = get_linear_schedule_with_warmup(
    optimizer, int(0.1 * total_steps), total_steps
)

In [10]:
# ── Cellule 9 : Training loop ─────────────────────────────────────────────
from sklearn.metrics import f1_score, classification_report
import os

OUT_DIR  = '/kaggle/working/deberta_v3_parent'
os.makedirs(OUT_DIR, exist_ok=True)
best_f1  = 0.0
best_acc = 0.0

for epoch in range(N_EPOCHS):
    model.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        lbl  = batch['labels'].to(DEVICE)
        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=mask).logits.float()
        loss   = loss_fn(logits, lbl)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        if (step+1) % 300 == 0:
            print(f'  [Ep {epoch+1}/{N_EPOCHS}] step {step+1}/{len(train_loader)} loss={total_loss/(step+1):.4f}')

    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(input_ids=batch['input_ids'].to(DEVICE),
                        attention_mask=batch['attention_mask'].to(DEVICE))
            preds_all.extend(out.logits.float().argmax(-1).cpu().tolist())
            labels_all.extend(batch['labels'].tolist())

    f1  = f1_score(labels_all, preds_all, average='macro', zero_division=0)
    acc = (np.array(preds_all) == np.array(labels_all)).mean()
    tag = '[BEST]' if f1 > best_f1 else ''
    print(f'Epoch {epoch+1}/{N_EPOCHS}  loss={total_loss/len(train_loader):.4f}  macro-F1={f1:.4f}  acc={acc:.4f}  {tag}')
    if f1 > best_f1:
        best_f1, best_acc = f1, acc
        model.save_pretrained(OUT_DIR)
        tokenizer.save_pretrained(OUT_DIR)

print(f'\n==== BEST  macro-F1={best_f1:.4f}  accuracy={best_acc:.4f} ({best_acc*100:.1f}%) ====')
print()
print(classification_report([CLASSES[l] for l in labels_all],
                             [CLASSES[p] for p in preds_all], zero_division=0))

  [Ep 1/5] step 300/2393 loss=1.1850
  [Ep 1/5] step 600/2393 loss=0.9778
  [Ep 1/5] step 900/2393 loss=0.8876
  [Ep 1/5] step 1200/2393 loss=0.8375
  [Ep 1/5] step 1500/2393 loss=0.8022
  [Ep 1/5] step 1800/2393 loss=0.7766
  [Ep 1/5] step 2100/2393 loss=0.7575
Epoch 1/5  loss=0.7400  macro-F1=0.7124  acc=0.7816  [BEST]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [Ep 2/5] step 300/2393 loss=0.5925
  [Ep 2/5] step 600/2393 loss=0.5898
  [Ep 2/5] step 900/2393 loss=0.5851
  [Ep 2/5] step 1200/2393 loss=0.5751
  [Ep 2/5] step 1500/2393 loss=0.5755
  [Ep 2/5] step 1800/2393 loss=0.5749
  [Ep 2/5] step 2100/2393 loss=0.5723
Epoch 2/5  loss=0.5718  macro-F1=0.7231  acc=0.7913  [BEST]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [Ep 3/5] step 300/2393 loss=0.5060
  [Ep 3/5] step 600/2393 loss=0.4988
  [Ep 3/5] step 900/2393 loss=0.5025
  [Ep 3/5] step 1200/2393 loss=0.5073
  [Ep 3/5] step 1500/2393 loss=0.5047
  [Ep 3/5] step 1800/2393 loss=0.5057
  [Ep 3/5] step 2100/2393 loss=0.5065
Epoch 3/5  loss=0.5077  macro-F1=0.7363  acc=0.7963  [BEST]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  [Ep 4/5] step 300/2393 loss=0.4597
  [Ep 4/5] step 600/2393 loss=0.4632
  [Ep 4/5] step 900/2393 loss=0.4582
  [Ep 4/5] step 1200/2393 loss=0.4583
  [Ep 4/5] step 1500/2393 loss=0.4578
  [Ep 4/5] step 1800/2393 loss=0.4533
  [Ep 4/5] step 2100/2393 loss=0.4515
Epoch 4/5  loss=0.4527  macro-F1=0.7288  acc=0.7955  
  [Ep 5/5] step 300/2393 loss=0.4193
  [Ep 5/5] step 600/2393 loss=0.4174
  [Ep 5/5] step 900/2393 loss=0.4156
  [Ep 5/5] step 1200/2393 loss=0.4114
  [Ep 5/5] step 1500/2393 loss=0.4090
  [Ep 5/5] step 1800/2393 loss=0.4092
  [Ep 5/5] step 2100/2393 loss=0.4060
Epoch 5/5  loss=0.4045  macro-F1=0.7300  acc=0.7955  

==== BEST  macro-F1=0.7363  accuracy=0.7963 (79.6%) ====

              precision    recall  f1-score   support

         Bug       0.72      0.69      0.71       671
 Improvement       0.66      0.72      0.69      1012
       Other       0.57      0.49      0.53       572
    Sub-task       0.99      1.00      1.00      1554

    accuracy                       

In [11]:
# ── Cellule 10 : Test inférence ───────────────────────────────────────────
model.eval()
tests = [
    ('[HAS-PARENT] [NO-DESCRIPTION] TICKET: Reduce toSeq in RDDOperationGraphWrapper', 'Sub-task'),
    ('[NO-PARENT] [BUG-SIGNAL] TICKET: NullPointerException in SparkContext', 'Bug'),
    ('[NO-PARENT] TICKET: optimize byteToString routines\nPRI: Minor', 'Improvement'),
    ('[HAS-PARENT] TICKET: Add function array_compact to connect module', 'Sub-task'),
]
print('Exemples inférence :')
for text, expected in tests:
    enc  = tokenizer(text, return_tensors='pt', truncation=True, max_length=256, padding=True).to(DEVICE)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits.float(), dim=-1)[0]
    pred = CLASSES[probs.argmax().item()]
    conf = probs.max().item()
    ok   = '✓' if pred == expected else '✗'
    print(f'  {ok} {pred:<15} ({conf*100:.0f}%)  attendu={expected:<15} | {text[:70]}')

Exemples inférence :
  ✓ Sub-task        (100%)  attendu=Sub-task        | [HAS-PARENT] [NO-DESCRIPTION] TICKET: Reduce toSeq in RDDOperationGrap
  ✓ Bug             (96%)  attendu=Bug             | [NO-PARENT] [BUG-SIGNAL] TICKET: NullPointerException in SparkContext
  ✓ Improvement     (93%)  attendu=Improvement     | [NO-PARENT] TICKET: optimize byteToString routines
PRI: Minor
  ✓ Sub-task        (100%)  attendu=Sub-task        | [HAS-PARENT] TICKET: Add function array_compact to connect module


In [1]:
# ── Cellule download : zipper et télécharger le modèle ───────────────────
import shutil, os
from IPython.display import FileLink

MODEL_DIR = '/kaggle/working/deberta_v3_parent'
ZIP_PATH  = '/kaggle/working/deberta_v3_parent.zip'

# Créer le zip
print("Création du zip...")
shutil.make_archive(
    '/kaggle/working/deberta_v3_parent',  # nom sans extension
    'zip',                                 # format
    MODEL_DIR                              # dossier à zipper
)

size_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f"Zip créé : {ZIP_PATH}  ({size_mb:.0f} MB)")

# Lien de téléchargement
display(FileLink('/kaggle/working/deberta_v3_parent.zip'))

Création du zip...
Zip créé : /kaggle/working/deberta_v3_parent.zip  (599 MB)


/kaggle/working/deberta_v3_parent.zip